Neural networks comprise of layers/modules that perform operations on data. The `torch.nn` namespace provides all the building blocks you need to build your own neural network. Every module in PyTorch subclasses the `nn.Module`. A neural network is a module itself that consists of other modules (layers). This nested structure allows for building and managing complex architectures easily.

In [1]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets,transforms

## Get Device for Training

We want to be able to train our model on an **accelerator** such as CUDA, MPS, MTIA, or XPU. If the current accelerator is available, we will use it. Otherwise, we use the CPU.

In [2]:
device=torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(device)

cuda


## Define the Class

We define our neural network by subclassing `nn.Module`, and initialize the neural network layers in `__init__`. Every `nn.Module` subclass implements the operations on input data in the `forward` method.

In [3]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten=nn.Flatten()   #[1,28,28] to [1,784] ;[batch_size,1,28,28] to [batch_size,784]; Flatten(start_dim=1, end_dim=-1)
        self.linear_relu_stack=nn.Sequential(
            nn.Linear(28*28,512),
            nn.ReLU(),
            nn.Linear(512,512),
            nn.ReLU(),
            nn.Linear(512,10)   #the shape of output is [batch_size,10], so dim=1 represent logits
        )

    def forward(self,x):
        x=self.flatten(x)
        logits=self.linear_relu_stack(x)
        return logits

In [4]:
#We create an instance of NeuralNetwork, and move it to the device, and print its structure.
model=NeuralNetwork().to(device)
print(model)    #call __repr__() method of nn.Module

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


To use the model, we pass it the input data. This executes the model’s `forward`, along with some background operations. Do not call `model.forward()` directly!

Calling the model on the input returns a 2-dimensional tensor with dim=0 corresponding to each output of 10 raw predicted values for each class, and dim=1 corresponding to the individual values of each output. We get the prediction probabilities by passing it through an instance of the `nn.Softmax` module.

In [5]:
x=torch.rand(1,28,28,device=device)
logit=model(x)
pred_probab=nn.Softmax(dim=1)(logit)    # Important : it equals softmax=nn.Softmax(dim=1)   pred_probab=softmax(logit)
y_pred=pred_probab.argmax(1)    #argmax(1) means find out the biggest value along dim=1
#max() returns the biggest value itself; argmax() returns the index of the biggest value
print(f"Predicted class:{y_pred}")

Predicted class:tensor([4], device='cuda:0')


## Model Layers

In [6]:
input_image=torch.rand(3,28,28)
print(input_image.size())

torch.Size([3, 28, 28])


## nn.Flatten

We initialize the `nn.Flatten` layer to convert each 2D 28x28 image into a contiguous array of 784 pixel values ( the minibatch dimension (at dim=0) is maintained).

In [7]:
flatten=nn.Flatten()
flat_image=flatten(input_image)
print(flat_image.size())

torch.Size([3, 784])


## nn.Linear

The linear layer is a module that applies a linear transformation on the input using its stored **weights** and **biases**.

In [8]:
layer1=nn.Linear(in_features=28*28,out_features=20)
hidden1=layer1(flat_image)
print(hidden1.size())

torch.Size([3, 20])


## nn.ReLU

Non-linear activations are what create the complex mappings between the model’s inputs and outputs. They are applied after linear transformations to introduce nonlinearity, helping neural networks learn a wide variety of phenomena.

In this model, we use `nn.ReLU` between our linear layers, but there’s other activations to introduce non-linearity in your model.

In [9]:
print(f"before ReLU:{hidden1}\n")
hidden1=nn.ReLU()(hidden1)
print(f"After ReLU:{hidden1}")

before ReLU:tensor([[ 0.0775, -1.0137, -0.2923,  0.2772,  0.2415,  0.5151, -0.4299,  0.2615,
          0.7393,  0.3461, -0.0443, -0.1860,  0.2515,  0.2106, -0.1482, -0.3575,
          0.0510, -0.1767, -0.3409, -0.0696],
        [ 0.2203, -0.2788, -0.4455,  0.3519, -0.1544,  0.6280, -0.4148,  0.3090,
          0.9257,  0.3016,  0.0522, -0.1029,  0.5269, -0.2400,  0.0960, -0.3655,
          0.1776, -0.1892, -0.4926, -0.1680],
        [ 0.0871, -0.5761, -0.2188,  0.1501, -0.2096,  0.4188, -0.1412,  0.3502,
          0.6262, -0.0547,  0.1876, -0.2166, -0.1253,  0.4601,  0.0472, -0.0339,
         -0.1658, -0.2007, -0.3007, -0.0288]], grad_fn=<AddmmBackward0>)

After ReLU:tensor([[0.0775, 0.0000, 0.0000, 0.2772, 0.2415, 0.5151, 0.0000, 0.2615, 0.7393,
         0.3461, 0.0000, 0.0000, 0.2515, 0.2106, 0.0000, 0.0000, 0.0510, 0.0000,
         0.0000, 0.0000],
        [0.2203, 0.0000, 0.0000, 0.3519, 0.0000, 0.6280, 0.0000, 0.3090, 0.9257,
         0.3016, 0.0522, 0.0000, 0.5269, 0.0000, 0.0960,

## nn.Sequential

`nn.Sequential` is an ordered container of modules. The data is passed through all the modules in the same order as defined. You can use sequential containers to put together a quick network like `seq_modules`.

In [10]:
seq_modules=nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20,10)
)
input_image=torch.rand(3,28,28)
logits=seq_modules(input_image)

## nn.Softmax

The last linear layer of the neural network returns logits - raw values in [-infty, infty] - which are passed to the `nn.Softmax` module. The logits are scaled to values [0, 1] representing the model’s predicted probabilities for each class. `dim` parameter indicates the dimension along which the values must sum to 1.

In [12]:
softmax=nn.Softmax(dim=1)
pred_probab=softmax(logits)

## Model Parameters

Many layers inside a neural network are parameterized, i.e. have associated weights and biases that are optimized during training. Subclassing `nn.Module` automatically tracks all fields defined inside your model object, and makes all parameters accessible using your model’s `parameters()` or `named_parameters()` methods.

In this example, we iterate over each parameter, and print its size and a preview of its values.

In [ ]:
print(f"Model structure:{model}\n")
for name,param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Value: {param[:2]}") #param[:2] extract first two values of dim=1
#the output of named_parameters() is (name,param). and "name" is the name of the parameter;"param" is the tensor of the parameter

Model structure:NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)

Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Value: tensor([[-0.0009, -0.0276, -0.0058,  ...,  0.0174,  0.0315,  0.0300],
        [-0.0128,  0.0007,  0.0219,  ..., -0.0017,  0.0326,  0.0321]],
       device='cuda:0', grad_fn=<SliceBackward0>)
Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Value: tensor([-0.0182, -0.0172], device='cuda:0', grad_fn=<SliceBackward0>)
Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Value: tensor([[ 0.0345,  0.0341,  0.0121,  ...,  0.0115,  0.0062, -0.0136],
        [-0.0249,  0.0404,  0.0047,  ...,  0.0241, -0.0089, -0.0125]],
       device='cuda:0', grad_fn=<SliceBackward0